# DeNICE audit training notebook

Kaggle entrypoint for the consolidated audit plan. Run one mode at a time; do not use raw checkpoints as continuation state.

In [ ]:
import os, subprocess, sys
from pathlib import Path

# Set this to a pushed audited commit before starting a Kaggle session.
GITHUB_REPO = 'https://github.com/khoilv2005/FL_IL_IDS.git'
GITHUB_REF = 'main'
REPO_PATH = Path('/kaggle/working/FL_IL_IDS')
DATA_DIR = '/kaggle/input/datasets/khoilv2005/100-clients/100-clients'

if REPO_PATH.exists():
    subprocess.run(['rm', '-rf', str(REPO_PATH)], check=True)
subprocess.run(['git', 'clone', GITHUB_REPO, str(REPO_PATH)], check=True)
subprocess.run(['git', '-C', str(REPO_PATH), 'checkout', GITHUB_REF], check=True)
commit = subprocess.check_output(['git', '-C', str(REPO_PATH), 'rev-parse', 'HEAD'], text=True).strip()
print('Audited code commit:', commit)
subprocess.run([sys.executable, '-c', 'import torch; print({\"cuda_available\": torch.cuda.is_available(), \"device_count\": torch.cuda.device_count()})'], check=True)
if not Path(DATA_DIR).is_dir():
    raise FileNotFoundError(DATA_DIR)


In [ ]:
# Choose exactly one: D1, D3, D4_BASE, D4, D5, D6, or FULL.
RUN_MODE = 'D6'
SEED = 43
EVAL_DEVICE = 'cuda'
OUTPUT_ROOT = Path(f'/kaggle/working/denice_{RUN_MODE.lower()}_seed_{SEED}')

# D4 only: an audited full continuation artifact after task 4.
# Never point this at checkpoint_task_4.pt.
D4_BASE_CONTINUATION = str(REPO_PATH / 'checkpoints/denice_d4_base_seed_42/continuation_state_task_4.pt')

# D5 only: selected D4 reserve_000 endpoint versioned in this repository via Git LFS.
D5_CHECKPOINT = str(REPO_PATH / 'checkpoints/denice_d4_seed_42/reserve_000/checkpoint_task_5_round_4.pt')

if EVAL_DEVICE == 'cuda':
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('This audit budget requires a Kaggle GPU accelerator.')


In [ ]:
env = {**os.environ, 'DENICE_REPO_DIR': str(REPO_PATH), 'DENICE_DATA_DIR': DATA_DIR,
       'DENICE_SEED': str(SEED), 'DENICE_EVAL_DEVICE': EVAL_DEVICE}
if RUN_MODE == 'D1':
    env['D1_OUTPUT_ROOT'] = str(OUTPUT_ROOT)
    command = [sys.executable, str(REPO_PATH / 'run_denice_d1_kaggle.py')]
elif RUN_MODE == 'D3':
    env['D3_OUTPUT_ROOT'] = str(OUTPUT_ROOT)
    command = [sys.executable, str(REPO_PATH / 'run_denice_d3_kaggle.py')]
elif RUN_MODE == 'D4_BASE':
    env['D4_BASE_OUTPUT_ROOT'] = str(OUTPUT_ROOT)
    command = [sys.executable, str(REPO_PATH / 'run_denice_d4_base_kaggle.py')]
elif RUN_MODE == 'D4':
    if not D4_BASE_CONTINUATION:
        raise ValueError('D4_BASE_CONTINUATION must be continuation_state_task_4.pt')
    env['D4_OUTPUT_ROOT'] = str(OUTPUT_ROOT)
    env['D4_BASE_CONTINUATION'] = D4_BASE_CONTINUATION
    command = [sys.executable, str(REPO_PATH / 'run_denice_d4_kaggle.py')]
elif RUN_MODE == 'D5':
    if not Path(D5_CHECKPOINT).is_file():
        raise FileNotFoundError('D5 needs the selected D4 reserve_000 final checkpoint; set D5_CHECKPOINT.')
    env['D5_OUTPUT_ROOT'] = str(OUTPUT_ROOT)
    env['D5_CHECKPOINT'] = D5_CHECKPOINT
    command = [sys.executable, str(REPO_PATH / 'run_denice_d5_kaggle.py')]
elif RUN_MODE == 'D6':
    env['D6_OUTPUT_ROOT'] = str(OUTPUT_ROOT)
    command = [sys.executable, str(REPO_PATH / 'run_denice_d6_kaggle.py')]
elif RUN_MODE == 'FULL':
    env.update({'DENICE_TRAIN_PHASE': '5', 'DENICE_OUTPUT_DIR': str(OUTPUT_ROOT)})
    command = [sys.executable, str(REPO_PATH / 'train_incremental_kaggle.py')]
else:
    raise ValueError(f'Unknown RUN_MODE: {RUN_MODE}')
print('Running:', ' '.join(command), '->', OUTPUT_ROOT)
subprocess.run(command, check=True, env=env, cwd=REPO_PATH)


In [ ]:
import json
for name in ('d1_decision_report.json', 'd3_decision_report.json', 'd4_base_manifest.json', 'd4_manifest.json', 'd5_manifest.json', 'd5_decision_report.json', 'd6_manifest.json', 'd6_decision_report.json'):
    path = OUTPUT_ROOT / name
    if path.is_file():
        print(f'\n=== {name} ===')
        print(json.dumps(json.loads(path.read_text(encoding='utf-8')), indent=2)[:12000])
print('Output root:', OUTPUT_ROOT)
